# Phase 2: Data Preparation

## OB-GYN Provider Network Analysis

---

### Objective

Transform raw datasets from Phase 1 into clean, analysis-ready files. This notebook handles the most computationally intensive step of the project — filtering 11 GB and 2.9 GB files down to the OB-GYN subset and classifying all providers in the network.

### Steps

1. **Load reference data** — NUCC taxonomy (full provider classification) and CDC WONDER exports
2. **Extract OB-GYN providers from NPPES** — Chunked scan of 11 GB file, filtering on 207V taxonomy codes across all 15 taxonomy slots
3. **Filter CMS edge list to OB-GYN pairs** — Chunked scan of 2.9 GB file, keeping only edges where at least one provider is OB-GYN
4. **Classify non-OB-GYN providers** — Second chunked NPPES pass to look up the specialty of every non-OB-GYN provider that appears in our edge list
5. **Enrich edge list with provider details** — Merge specialty, city, state, ZIP onto both sides of each provider pair
6. **Compute county-level maternal morbidity rates** — Join CDC WONDER birth and morbidity counts
7. **Export clean datasets** to `data/outputs/`

### Outputs

| File | Description |
|------|-------------|
| `obgyn_providers.csv` | All OB-GYN NPIs with practice location and sub-specialty |
| `other_providers.csv` | Non-OB-GYN NPIs that appear in the edge list, with their NUCC classification and specialization |
| `obgyn_edges.csv` | CMS shared patient pairs (at least one OB-GYN) with full provider details on both sides |
| `county_morbidity_rates.csv` | County-level maternal morbidity rates (morbidity births / total births) |

### Performance Notes

- NPPES (11 GB, ~8M rows): Two chunked passes — first for OB-GYN extraction, second for non-OB-GYN lookup
- CMS (2.9 GB, ~49M rows): Read in chunks of 500,000 rows, filtering against the OB-GYN NPI set
- Expected runtime: 15–30 minutes depending on disk speed

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

print(f"Notebook executed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Notebook executed: 2026-02-22 18:14:43
pandas version: 2.3.3
numpy version: 1.26.4


In [2]:
# -------------------------------------------------------------------
# FILE PATHS
# -------------------------------------------------------------------

RAW_DIR = Path("../data/inputs/raw")
OUTPUT_DIR = Path("../data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs (from Phase 1)
CMS_FILE = RAW_DIR / "cms_shared_patient" / "pspp2015_60.csv"
NPPES_FILE = RAW_DIR / "nppes" / "npidata_pfile_20050523-20260208.csv"
NUCC_FILE = RAW_DIR / "nucc_taxonomy" / "nucc_taxonomy_151.csv"
CDC_BIRTHS_FILE = RAW_DIR / "cdc_wonder" / "cdc_wonder_2016_births_by_county.csv"
CDC_MORBIDITY_FILE = RAW_DIR / "cdc_wonder" / "cdc_wonder_2016_maternal_morbidity_by_county.csv"

# Outputs
OBGYN_PROVIDERS_OUT = OUTPUT_DIR / "obgyn_providers.csv"
OTHER_PROVIDERS_OUT = OUTPUT_DIR / "other_providers.csv"
OBGYN_EDGES_OUT = OUTPUT_DIR / "obgyn_edges.csv"
COUNTY_MORBIDITY_OUT = OUTPUT_DIR / "county_morbidity_rates.csv"

# Verify inputs exist
for name, fpath in [("CMS", CMS_FILE), ("NPPES", NPPES_FILE), ("NUCC", NUCC_FILE),
                     ("CDC Births", CDC_BIRTHS_FILE), ("CDC Morbidity", CDC_MORBIDITY_FILE)]:
    status = "FOUND" if fpath.exists() else "MISSING"
    print(f"  {name:<20} {status}")

print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")

  CMS                  FOUND
  NPPES                FOUND
  NUCC                 FOUND
  CDC Births           FOUND
  CDC Morbidity        FOUND

Output directory: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/network_analysis/maternal_network/data/outputs


---

## 2. Load Reference Data

### 2a. NUCC Taxonomy — OB-GYN Codes (207V Family)

In [3]:
# -------------------------------------------------------------------
# LOAD: NUCC Taxonomy — full lookup + OB-GYN codes
# -------------------------------------------------------------------

nucc_df = pd.read_csv(NUCC_FILE, dtype=str)

# --- Full taxonomy lookup (for classifying ALL providers later) ---
# Build a dictionary: taxonomy_code -> (Type, Classification, Specialization)
NUCC_LOOKUP = {}
for _, row in nucc_df.iterrows():
    code = row["Code"]
    NUCC_LOOKUP[code] = {
        "type": row["Type"] if pd.notna(row["Type"]) else "",
        "classification": row["Classification"] if pd.notna(row["Classification"]) else "",
        "specialization": row["Specialization"] if pd.notna(row["Specialization"]) else ""
    }

print(f"Full NUCC taxonomy: {len(NUCC_LOOKUP):,} codes loaded")

# --- OB-GYN subset (207V family) ---
obgyn_taxonomy = nucc_df[nucc_df["Code"].str.startswith("207V", na=False)].copy()

# Build a lookup dictionary: taxonomy code -> sub-specialty label
obgyn_taxonomy["label"] = obgyn_taxonomy.apply(
    lambda r: r["Specialization"] if pd.notna(r["Specialization"]) else r["Classification"],
    axis=1
)
OBGYN_CODE_SET = set(obgyn_taxonomy["Code"])
OBGYN_CODE_LABELS = dict(zip(obgyn_taxonomy["Code"], obgyn_taxonomy["label"]))

print(f"OB-GYN taxonomy codes: {len(OBGYN_CODE_SET)}")
print()
for code, label in sorted(OBGYN_CODE_LABELS.items()):
    print(f"  {code}  {label}")

Full NUCC taxonomy: 838 codes loaded
OB-GYN taxonomy codes: 10

  207V00000X  Obstetrics & Gynecology
  207VB0002X  Obesity Medicine
  207VC0200X  Critical Care Medicine
  207VE0102X  Reproductive Endocrinology
  207VF0040X  Female Pelvic Medicine and Reconstructive Surgery
  207VG0400X  Gynecology
  207VH0002X  Hospice and Palliative Medicine
  207VM0101X  Maternal & Fetal Medicine
  207VX0000X  Obstetrics
  207VX0201X  Gynecologic Oncology


### 2b. CDC WONDER — Births & Maternal Morbidity by County

In [4]:
# -------------------------------------------------------------------
# HELPER: Load CDC WONDER export (same as Phase 1)
# -------------------------------------------------------------------

def load_cdc_wonder(filepath):
    """Load a CDC WONDER export, stopping at the metadata separator."""
    lines = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('"---'):
                break
            lines.append(line)
    
    # Detect delimiter from header line
    header = lines[0]
    delimiter = '\t' if '\t' in header else ','
    
    return pd.read_csv(StringIO(''.join(lines)), sep=delimiter, dtype=str)

In [5]:
# -------------------------------------------------------------------
# LOAD: CDC WONDER exports
# -------------------------------------------------------------------

cdc_births = load_cdc_wonder(CDC_BIRTHS_FILE)
cdc_morbidity = load_cdc_wonder(CDC_MORBIDITY_FILE)

print(f"CDC Births:    {len(cdc_births):,} rows, columns: {list(cdc_births.columns)}")
print(f"CDC Morbidity: {len(cdc_morbidity):,} rows, columns: {list(cdc_morbidity.columns)}")

CDC Births:    627 rows, columns: ['Notes', 'County of Residence', 'County of Residence Code', 'Births']
CDC Morbidity: 578 rows, columns: ['Notes', 'County of Residence', 'County of Residence Code', 'Births']


---

## 3. Extract OB-GYN Providers from NPPES

The NPPES file is ~11 GB with ~8 million provider records and 330 columns. We read it in chunks of 100,000 rows, loading only the 22 columns we need. For each chunk, we check all 15 taxonomy code slots — a provider is flagged as OB-GYN if **any** of their taxonomy codes starts with `207V`.

**Filtering logic:**
- Entity Type Code = `1` (Individual providers only; organizations excluded)
- At least one of `Healthcare Provider Taxonomy Code_1` through `_15` is in the 207V family

In [6]:
# -------------------------------------------------------------------
# DEFINE: NPPES columns to load
# -------------------------------------------------------------------

TAXONOMY_COLS = [f"Healthcare Provider Taxonomy Code_{i}" for i in range(1, 16)]

NPPES_USE_COLS = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
] + TAXONOMY_COLS

print(f"Loading {len(NPPES_USE_COLS)} columns from NPPES...")

Loading 22 columns from NPPES...


In [7]:
# -------------------------------------------------------------------
# EXTRACT: OB-GYN providers via chunked read
# -------------------------------------------------------------------
# This cell processes ~11 GB in chunks. Expected runtime: 5-10 minutes.
# -------------------------------------------------------------------

CHUNK_SIZE = 100_000
obgyn_chunks = []
total_rows = 0
total_obgyn = 0

t0 = time.time()
print(f"Scanning NPPES file ({NPPES_FILE.stat().st_size / (1024**3):.1f} GB)...")
print(f"Chunk size: {CHUNK_SIZE:,} rows")
print()

reader = pd.read_csv(
    NPPES_FILE,
    usecols=NPPES_USE_COLS,
    dtype=str,
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)
    
    # Filter to individual providers only (Entity Type Code = 1)
    individuals = chunk[chunk["Entity Type Code"] == "1"].copy()
    
    if len(individuals) == 0:
        continue
    
    # Check if ANY of the 15 taxonomy code slots contains a 207V code
    is_obgyn = pd.Series(False, index=individuals.index)
    for col in TAXONOMY_COLS:
        is_obgyn = is_obgyn | individuals[col].str.startswith("207V", na=False)
    
    obgyn_in_chunk = individuals[is_obgyn]
    
    if len(obgyn_in_chunk) > 0:
        obgyn_chunks.append(obgyn_in_chunk)
        total_obgyn += len(obgyn_in_chunk)
    
    # Progress update every 10 chunks
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        rate = total_rows / elapsed
        print(f"  Chunk {i+1:>4}: {total_rows:>10,} rows scanned | "
              f"{total_obgyn:>6,} OB-GYN found | "
              f"{elapsed:.0f}s elapsed | {rate:,.0f} rows/s")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")
print(f"Total rows scanned: {total_rows:,}")
print(f"OB-GYN providers found: {total_obgyn:,}")

Scanning NPPES file (10.4 GB)...
Chunk size: 100,000 rows

  Chunk   10:  1,000,000 rows scanned | 22,337 OB-GYN found | 11s elapsed | 93,251 rows/s
  Chunk   20:  2,000,000 rows scanned | 33,268 OB-GYN found | 22s elapsed | 91,002 rows/s
  Chunk   30:  3,000,000 rows scanned | 41,132 OB-GYN found | 32s elapsed | 92,366 rows/s
  Chunk   40:  4,000,000 rows scanned | 46,997 OB-GYN found | 44s elapsed | 90,814 rows/s
  Chunk   50:  5,000,000 rows scanned | 51,976 OB-GYN found | 56s elapsed | 89,891 rows/s
  Chunk   60:  6,000,000 rows scanned | 56,069 OB-GYN found | 67s elapsed | 89,247 rows/s
  Chunk   70:  7,000,000 rows scanned | 58,897 OB-GYN found | 79s elapsed | 88,499 rows/s
  Chunk   80:  8,000,000 rows scanned | 59,720 OB-GYN found | 91s elapsed | 88,231 rows/s
  Chunk   90:  9,000,000 rows scanned | 60,337 OB-GYN found | 103s elapsed | 87,369 rows/s

Done in 107.4s
Total rows scanned: 9,368,082
OB-GYN providers found: 60,422


In [8]:
# -------------------------------------------------------------------
# COMBINE: Merge all OB-GYN chunks into a single DataFrame
# -------------------------------------------------------------------

obgyn_providers_raw = pd.concat(obgyn_chunks, ignore_index=True)
print(f"Combined OB-GYN providers: {len(obgyn_providers_raw):,} rows")

# Verify no duplicate NPIs
n_dupes = obgyn_providers_raw["NPI"].duplicated().sum()
print(f"Duplicate NPIs: {n_dupes:,}")
if n_dupes > 0:
    print("Dropping duplicates (keeping first occurrence)...")
    obgyn_providers_raw = obgyn_providers_raw.drop_duplicates(subset="NPI", keep="first")
    print(f"After dedup: {len(obgyn_providers_raw):,} rows")

Combined OB-GYN providers: 60,422 rows
Duplicate NPIs: 0


In [9]:
# -------------------------------------------------------------------
# RESHAPE: Identify primary OB-GYN taxonomy code and sub-specialty
# -------------------------------------------------------------------
# A provider may have multiple taxonomy codes across 15 slots.
# We identify the FIRST 207V code found (reading slots 1-15 in order)
# as their "primary" OB-GYN taxonomy code.
# -------------------------------------------------------------------

def get_primary_obgyn_code(row):
    """Return the first 207V taxonomy code found across all 15 slots."""
    for col in TAXONOMY_COLS:
        val = row[col]
        if pd.notna(val) and val.startswith("207V"):
            return val
    return None

obgyn_providers_raw["primary_obgyn_code"] = obgyn_providers_raw.apply(
    get_primary_obgyn_code, axis=1
)

# Map to human-readable sub-specialty label
obgyn_providers_raw["obgyn_subspecialty"] = (
    obgyn_providers_raw["primary_obgyn_code"].map(OBGYN_CODE_LABELS)
)

print("Sub-specialty distribution:")
print(obgyn_providers_raw["obgyn_subspecialty"].value_counts().to_string())

Sub-specialty distribution:
obgyn_subspecialty
Obstetrics & Gynecology                              48586
Gynecology                                            4115
Maternal & Fetal Medicine                             2243
Obstetrics                                            1633
Reproductive Endocrinology                            1490
Gynecologic Oncology                                  1477
Female Pelvic Medicine and Reconstructive Surgery      656
Hospice and Palliative Medicine                        100
Critical Care Medicine                                  41
Obesity Medicine                                        25


In [10]:
# -------------------------------------------------------------------
# CLEAN: Rename columns for downstream use and extract 5-digit ZIP
# -------------------------------------------------------------------

obgyn_providers = obgyn_providers_raw[[
    "NPI",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "primary_obgyn_code",
    "obgyn_subspecialty",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
]].copy()

obgyn_providers.columns = [
    "npi", "last_name", "first_name", "taxonomy_code", "subspecialty",
    "city", "state", "zip_raw"
]

# Extract 5-digit ZIP (NPPES may include ZIP+4 as 9 digits)
obgyn_providers["zip5"] = obgyn_providers["zip_raw"].str[:5]

print(f"OB-GYN provider table: {len(obgyn_providers):,} rows x {len(obgyn_providers.columns)} cols")
print(f"\nColumns: {list(obgyn_providers.columns)}")
print(f"\nState distribution (top 10):")
print(obgyn_providers["state"].value_counts().head(10).to_string())
print(f"\nSample:")
display(obgyn_providers.head(10))

OB-GYN provider table: 60,422 rows x 9 cols

Columns: ['npi', 'last_name', 'first_name', 'taxonomy_code', 'subspecialty', 'city', 'state', 'zip_raw', 'zip5']

State distribution (top 10):
state
CA    6655
TX    4611
NY    4474
FL    3628
PA    2642
IL    2586
MI    2125
OH    2081
NC    2010
GA    1905

Sample:


,npi,last_name,first_name,taxonomy_code,subspecialty,city,state,zip_raw,zip5
0,1487657532,HUEBERT,ALLISON,207V00000X,Obstetrics & Gynecology,TULSA,OK,741352527,74135
1,1396748349,GRUNERT,GEORGE,207VE0102X,Reproductive Endocrinology,HOUSTON,TX,770632313,77063
2,1861495814,SMITH,FRANCISCO,207VX0201X,Gynecologic Oncology,NAPLES,FL,341101497,34110
3,1003819046,NYLANDER,BARBARA,207VG0400X,Gynecology,NASHVILLE,TN,372031513,37203
4,1932102969,HELFER,TAMARA,207V00000X,Obstetrics & Gynecology,CARMEL,IN,460323008,46032
5,1578566501,LA,BAO QUOC,207V00000X,Obstetrics & Gynecology,HOUSTON,TX,77024,77024
6,1922001957,PRESLEY,RICHARD,207V00000X,Obstetrics & Gynecology,NASHVILLE,TN,372032023,37203
7,1649273673,KOPCZYNSKI,TODD,207V00000X,Obstetrics & Gynecology,CHARLOTTE,NC,282179422,28217
8,1760485817,MORAN,SAM,207V00000X,Obstetrics & Gynecology,NASHVILLE,TN,372031838,37203
9,1811990864,GULLETT,JAMES,207V00000X,Obstetrics & Gynecology,HOUSTON,TX,770542935,77054


In [11]:
# -------------------------------------------------------------------
# BUILD: NPI set for CMS filtering
# -------------------------------------------------------------------

OBGYN_NPI_SET = set(obgyn_providers["npi"])
print(f"OB-GYN NPI set size: {len(OBGYN_NPI_SET):,}")
print(f"Sample NPIs: {list(OBGYN_NPI_SET)[:5]}")

OB-GYN NPI set size: 60,422
Sample NPIs: ['1083619852', '1033266879', '1699424028', '1639594641', '1184663585']


---

## 4. Filter CMS Edge List to OB-GYN Pairs

The CMS Shared Patient Patterns file has ~49 million provider pair rows. We scan it in chunks and keep only rows where **at least one** of the two NPIs (`npi1` or `npi2`) is in our OB-GYN provider set. This captures:

- **OB-GYN ↔ OB-GYN pairs** — referrals within the OB-GYN specialty
- **OB-GYN ↔ Non-OB-GYN pairs** — cross-specialty referrals (e.g., OB-GYN to MFM, to anesthesiology, to neonatology)

We also add a flag column `edge_type` to distinguish these two categories for downstream analysis.

In [12]:
# -------------------------------------------------------------------
# FILTER: CMS edges via chunked read
# -------------------------------------------------------------------
# This cell processes ~2.9 GB in chunks. Expected runtime: 3-8 minutes.
# -------------------------------------------------------------------

CMS_CHUNK_SIZE = 500_000
edge_chunks = []
total_rows = 0
total_kept = 0

t0 = time.time()
print(f"Scanning CMS Shared Patient Patterns ({CMS_FILE.stat().st_size / (1024**3):.1f} GB)...")
print(f"Chunk size: {CMS_CHUNK_SIZE:,} rows")
print(f"Filtering against {len(OBGYN_NPI_SET):,} OB-GYN NPIs")
print()

reader = pd.read_csv(
    CMS_FILE,
    dtype={"npi1": str, "npi2": str},
    chunksize=CMS_CHUNK_SIZE
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)
    
    # Keep rows where at least one NPI is OB-GYN
    npi1_match = chunk["npi1"].isin(OBGYN_NPI_SET)
    npi2_match = chunk["npi2"].isin(OBGYN_NPI_SET)
    matched = chunk[npi1_match | npi2_match].copy()
    
    if len(matched) > 0:
        # Classify edge type
        both_obgyn = matched["npi1"].isin(OBGYN_NPI_SET) & matched["npi2"].isin(OBGYN_NPI_SET)
        matched["edge_type"] = np.where(both_obgyn, "obgyn_obgyn", "obgyn_other")
        edge_chunks.append(matched)
        total_kept += len(matched)
    
    # Progress update every 10 chunks
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        rate = total_rows / elapsed
        print(f"  Chunk {i+1:>4}: {total_rows:>12,} rows scanned | "
              f"{total_kept:>8,} edges kept | "
              f"{elapsed:.0f}s elapsed | {rate:,.0f} rows/s")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")
print(f"Total rows scanned: {total_rows:,}")
print(f"OB-GYN edges kept: {total_kept:,} ({total_kept/total_rows*100:.2f}%)")

Scanning CMS Shared Patient Patterns (2.8 GB)...
Chunk size: 500,000 rows
Filtering against 60,422 OB-GYN NPIs

  Chunk   10:    5,000,000 rows scanned |   25,133 edges kept | 7s elapsed | 725,455 rows/s
  Chunk   20:   10,000,000 rows scanned |   51,209 edges kept | 13s elapsed | 757,389 rows/s
  Chunk   30:   15,000,000 rows scanned |   77,682 edges kept | 19s elapsed | 775,187 rows/s
  Chunk   40:   20,000,000 rows scanned |  104,427 edges kept | 27s elapsed | 738,876 rows/s
  Chunk   50:   25,000,000 rows scanned |  128,631 edges kept | 33s elapsed | 749,264 rows/s
  Chunk   60:   30,000,000 rows scanned |  155,012 edges kept | 39s elapsed | 763,246 rows/s
  Chunk   70:   35,000,000 rows scanned |  178,572 edges kept | 45s elapsed | 778,447 rows/s
  Chunk   80:   40,000,000 rows scanned |  201,004 edges kept | 51s elapsed | 782,942 rows/s
  Chunk   90:   45,000,000 rows scanned |  223,448 edges kept | 57s elapsed | 789,986 rows/s

Done in 62.7s
Total rows scanned: 49,335,361
OB-GYN

In [13]:
# -------------------------------------------------------------------
# COMBINE: Merge all edge chunks
# -------------------------------------------------------------------

obgyn_edges_raw = pd.concat(edge_chunks, ignore_index=True)
print(f"Combined OB-GYN edges: {len(obgyn_edges_raw):,} rows")

# Edge type distribution
print(f"\nEdge type distribution:")
print(obgyn_edges_raw["edge_type"].value_counts().to_string())

# Basic statistics
print(f"\nShared patient statistics (benecount):")
print(obgyn_edges_raw["benecount"].describe().to_string())

print(f"\nUnique NPIs in edge list:")
unique_npi1 = set(obgyn_edges_raw["npi1"])
unique_npi2 = set(obgyn_edges_raw["npi2"])
all_edge_npis = unique_npi1 | unique_npi2
print(f"  npi1 unique: {len(unique_npi1):,}")
print(f"  npi2 unique: {len(unique_npi2):,}")
print(f"  Combined unique: {len(all_edge_npis):,}")
print(f"  Of which are OB-GYN: {len(all_edge_npis & OBGYN_NPI_SET):,}")

Combined OB-GYN edges: 243,052 rows

Edge type distribution:
edge_type
obgyn_other    239300
obgyn_obgyn      3752

Shared patient statistics (benecount):
count    243052.000000
mean         28.196020
std          33.943759
min          11.000000
25%          13.000000
50%          18.000000
75%          29.000000
max        2426.000000

Unique NPIs in edge list:
  npi1 unique: 61,720
  npi2 unique: 62,881
  Combined unique: 71,955
  Of which are OB-GYN: 25,427


---

## 5. Classify Non-OB-GYN Providers

We now know which non-OB-GYN NPIs appear in our edge list (i.e., they share patients with at least one OB-GYN). To classify them by specialty, we do a **second chunked pass** through the NPPES file, this time looking up only those non-OB-GYN NPIs and extracting their primary taxonomy code. We then map that taxonomy code to the NUCC classification hierarchy (Type → Classification → Specialization).

In [14]:
# -------------------------------------------------------------------
# IDENTIFY: Non-OB-GYN NPIs that need classification
# -------------------------------------------------------------------

# Collect all unique NPIs from edges that are NOT in our OB-GYN set
all_edge_npis = set(obgyn_edges_raw["npi1"]) | set(obgyn_edges_raw["npi2"])
non_obgyn_npis_needed = all_edge_npis - OBGYN_NPI_SET

print(f"Total unique NPIs in edge list:   {len(all_edge_npis):,}")
print(f"Of which are OB-GYN:              {len(all_edge_npis & OBGYN_NPI_SET):,}")
print(f"Non-OB-GYN NPIs to look up:       {len(non_obgyn_npis_needed):,}")

Total unique NPIs in edge list:   71,955
Of which are OB-GYN:              25,427
Non-OB-GYN NPIs to look up:       46,528


In [15]:
# -------------------------------------------------------------------
# EXTRACT: Non-OB-GYN provider details via second chunked NPPES pass
# -------------------------------------------------------------------
# This is similar to the OB-GYN pass but we filter on a specific set
# of ~46K non-OB-GYN NPIs instead of taxonomy codes.
# Expected runtime: 5-10 minutes.
# -------------------------------------------------------------------

other_chunks = []
total_rows = 0
total_found = 0

t0 = time.time()
print(f"Second NPPES pass: looking up {len(non_obgyn_npis_needed):,} non-OB-GYN NPIs...")
print(f"Chunk size: {CHUNK_SIZE:,} rows")
print()

reader = pd.read_csv(
    NPPES_FILE,
    usecols=NPPES_USE_COLS,
    dtype=str,
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)
    
    # Filter to NPIs in our non-OB-GYN set
    matched = chunk[chunk["NPI"].isin(non_obgyn_npis_needed)]
    
    if len(matched) > 0:
        other_chunks.append(matched)
        total_found += len(matched)
    
    # Progress update every 10 chunks
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        rate = total_rows / elapsed
        print(f"  Chunk {i+1:>4}: {total_rows:>10,} rows scanned | "
              f"{total_found:>6,} found | "
              f"{elapsed:.0f}s elapsed | {rate:,.0f} rows/s")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")
print(f"Total rows scanned: {total_rows:,}")
print(f"Non-OB-GYN providers found: {total_found:,}")
print(f"Not found in NPPES: {len(non_obgyn_npis_needed) - total_found:,} (may be organizations or deactivated NPIs)")

Second NPPES pass: looking up 46,528 non-OB-GYN NPIs...
Chunk size: 100,000 rows

  Chunk   10:  1,000,000 rows scanned | 28,485 found | 10s elapsed | 103,862 rows/s
  Chunk   20:  2,000,000 rows scanned | 37,147 found | 19s elapsed | 106,545 rows/s
  Chunk   30:  3,000,000 rows scanned | 43,491 found | 28s elapsed | 107,315 rows/s
  Chunk   40:  4,000,000 rows scanned | 45,849 found | 37s elapsed | 107,326 rows/s
  Chunk   50:  5,000,000 rows scanned | 46,527 found | 47s elapsed | 107,300 rows/s
  Chunk   60:  6,000,000 rows scanned | 46,527 found | 56s elapsed | 107,234 rows/s
  Chunk   70:  7,000,000 rows scanned | 46,527 found | 65s elapsed | 106,957 rows/s
  Chunk   80:  8,000,000 rows scanned | 46,527 found | 75s elapsed | 107,003 rows/s
  Chunk   90:  9,000,000 rows scanned | 46,527 found | 84s elapsed | 107,283 rows/s

Done in 87.3s
Total rows scanned: 9,368,082
Non-OB-GYN providers found: 46,527
Not found in NPPES: 1 (may be organizations or deactivated NPIs)


In [16]:
# -------------------------------------------------------------------
# CLASSIFY: Map non-OB-GYN providers to NUCC taxonomy
# -------------------------------------------------------------------

other_providers_raw = pd.concat(other_chunks, ignore_index=True)

# Drop duplicates (a provider may appear in multiple chunks)
other_providers_raw = other_providers_raw.drop_duplicates(subset="NPI", keep="first")
print(f"Unique non-OB-GYN providers: {len(other_providers_raw):,}")

# Get primary taxonomy code (first non-null taxonomy slot)
def get_primary_taxonomy(row):
    """Return the first taxonomy code found across all 15 slots."""
    for col in TAXONOMY_COLS:
        val = row[col]
        if pd.notna(val) and val.strip():
            return val.strip()
    return None

other_providers_raw["primary_taxonomy_code"] = other_providers_raw.apply(
    get_primary_taxonomy, axis=1
)

# Map to NUCC classification hierarchy
other_providers_raw["nucc_type"] = other_providers_raw["primary_taxonomy_code"].apply(
    lambda x: NUCC_LOOKUP.get(x, {}).get("type", "Unknown") if pd.notna(x) else "Unknown"
)
other_providers_raw["nucc_classification"] = other_providers_raw["primary_taxonomy_code"].apply(
    lambda x: NUCC_LOOKUP.get(x, {}).get("classification", "Unknown") if pd.notna(x) else "Unknown"
)
other_providers_raw["nucc_specialization"] = other_providers_raw["primary_taxonomy_code"].apply(
    lambda x: NUCC_LOOKUP.get(x, {}).get("specialization", "") if pd.notna(x) else ""
)

print(f"\nTop 20 provider types (NUCC Type):")
print(other_providers_raw["nucc_type"].value_counts().head(20).to_string())

print(f"\nTop 20 classifications:")
print(other_providers_raw["nucc_classification"].value_counts().head(20).to_string())

Unique non-OB-GYN providers: 46,527

Top 20 provider types (NUCC Type):
nucc_type
Allopathic & Osteopathic Physicians                                             32242
Physician Assistants & Advanced Practice Nursing Providers                       2918
Hospitals                                                                        2675
Ambulatory Health Care Facilities                                                1778
Unknown                                                                          1419
Other Service Providers                                                          1140
Laboratories                                                                      831
Podiatric Medicine & Surgery Service Providers                                    692
Transportation Services                                                           657
Agencies                                                                          518
Eye and Vision Services Providers                         

In [17]:
# -------------------------------------------------------------------
# CLEAN: Build other_providers table for export and merge
# -------------------------------------------------------------------

other_providers = other_providers_raw[[
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "primary_taxonomy_code",
    "nucc_type",
    "nucc_classification",
    "nucc_specialization",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
]].copy()

other_providers.columns = [
    "npi", "entity_type_code", "last_name", "first_name", "taxonomy_code",
    "provider_type", "classification", "specialization",
    "city", "state", "zip_raw"
]

# Label entity types: 1 = Individual, 2 = Organization
other_providers["entity_type"] = other_providers["entity_type_code"].map({
    "1": "Individual", "2": "Organization"
}).fillna("Unknown")

# Extract 5-digit ZIP
other_providers["zip5"] = other_providers["zip_raw"].str[:5]

print(f"Non-OB-GYN provider table: {len(other_providers):,} rows x {len(other_providers.columns)} cols")
print(f"\nEntity type breakdown:")
print(other_providers["entity_type"].value_counts().to_string())
print(f"\nIndividuals: {(other_providers['entity_type']=='Individual').sum():,}")
print(f"Organizations: {(other_providers['entity_type']=='Organization').sum():,}")
print(f"\nSample:")
display(other_providers.head(10))

Non-OB-GYN provider table: 46,527 rows x 13 cols

Entity type breakdown:
entity_type
Individual      37114
Organization     8031
Unknown          1382

Individuals: 37,114
Organizations: 8,031

Sample:


,npi,entity_type_code,last_name,first_name,taxonomy_code,provider_type,classification,specialization,city,state,zip_raw,entity_type,zip5
0,1306849450,NaN,NaN,NaN,None,Unknown,Unknown,,NaN,NaN,NaN,Unknown,NaN
1,1114920261,1,THACKER,RICHARD,207R00000X,Allopathic & Osteopathic Physicians,Internal Medicine,,TALLAHASSEE,FL,323088417,Individual,32308
2,1073516027,1,CHANG,JOHN,207W00000X,Allopathic & Osteopathic Physicians,Ophthalmology,,PEARL RIVER,NY,109652029,Individual,10965
3,1033112081,1,FOGARTY,JOHN,207RP1001X,Allopathic & Osteopathic Physicians,Internal Medicine,Pulmonary Disease,GREENVILLE,NC,278345704,Individual,27834
4,1346243375,2,NaN,NaN,282N00000X,Hospitals,General Acute Care Hospital,,NEW BRUNSWICK,NJ,089011928,Organization,08901
5,1962405019,1,HAMAMI,ANWAR,207P00000X,Allopathic & Osteopathic Physicians,Emergency Medicine,,HUDSON,OH,44236,Individual,44236
6,1821091877,1,HASKELL,SUSAN,152W00000X,Eye and Vision Services Providers,Optometrist,,PORTSMOUTH,NH,038017156,Individual,03801
7,1972506921,1,FAGUNDUS,DUNCAN,207RR0500X,Allopathic & Osteopathic Physicians,Internal Medicine,Rheumatology,GREENVILLE,NC,278345704,Individual,27834
8,1821091869,1,BASS,PAOLA,207R00000X,Allopathic & Osteopathic Physicians,Internal Medicine,,DOTHAN,AL,363031928,Individual,36303
9,1497758437,NaN,NaN,NaN,None,Unknown,Unknown,,NaN,NaN,NaN,Unknown,NaN


---

## 6. Enrich Edge List with Provider Details

Merge provider details onto both sides of the edge list:
- **OB-GYN providers**: sub-specialty, city, state, ZIP from the OB-GYN table
- **Non-OB-GYN providers**: NUCC type, classification, specialization, city, state, ZIP from the other providers table

This means every edge now has full detail on BOTH sides — we can see exactly what type of specialist the OB-GYN is sharing patients with.

In [18]:
# -------------------------------------------------------------------
# MERGE: Provider details onto edge list (both OB-GYN and non-OB-GYN)
# -------------------------------------------------------------------

# --- OB-GYN lookup ---
obgyn_lookup = obgyn_providers[[
    "npi", "subspecialty", "city", "state", "zip5"
]].copy()

# --- Non-OB-GYN lookup (includes entity_type) ---
other_lookup = other_providers[[
    "npi", "entity_type", "provider_type", "classification", "specialization", "city", "state", "zip5"
]].copy()

# Start with raw edges
obgyn_edges = obgyn_edges_raw.copy()

# --- Merge OB-GYN details for npi1 ---
obgyn_edges = obgyn_edges.merge(
    obgyn_lookup.rename(columns={
        "npi": "npi1", "subspecialty": "npi1_subspecialty",
        "city": "npi1_city", "state": "npi1_state", "zip5": "npi1_zip5"
    }),
    on="npi1", how="left"
)

# --- Merge OB-GYN details for npi2 ---
obgyn_edges = obgyn_edges.merge(
    obgyn_lookup.rename(columns={
        "npi": "npi2", "subspecialty": "npi2_subspecialty",
        "city": "npi2_city", "state": "npi2_state", "zip5": "npi2_zip5"
    }),
    on="npi2", how="left"
)

# --- Merge non-OB-GYN details for npi1 ---
obgyn_edges = obgyn_edges.merge(
    other_lookup.rename(columns={
        "npi": "npi1", "entity_type": "npi1_entity_type",
        "provider_type": "npi1_other_type",
        "classification": "npi1_other_classification",
        "specialization": "npi1_other_specialization",
        "city": "npi1_other_city", "state": "npi1_other_state", "zip5": "npi1_other_zip5"
    }),
    on="npi1", how="left"
)

# --- Merge non-OB-GYN details for npi2 ---
obgyn_edges = obgyn_edges.merge(
    other_lookup.rename(columns={
        "npi": "npi2", "entity_type": "npi2_entity_type",
        "provider_type": "npi2_other_type",
        "classification": "npi2_other_classification",
        "specialization": "npi2_other_specialization",
        "city": "npi2_other_city", "state": "npi2_other_state", "zip5": "npi2_other_zip5"
    }),
    on="npi2", how="left"
)

# --- Consolidate: for each side, pick OB-GYN detail if available, else non-OB-GYN ---
for side in ["npi1", "npi2"]:
    obgyn_edges[f"{side}_specialty"] = obgyn_edges[f"{side}_subspecialty"].fillna(
        obgyn_edges[f"{side}_other_classification"]
    )
    for field in ["city", "state", "zip5"]:
        obgyn_edges[f"{side}_{field}_final"] = obgyn_edges[f"{side}_{field}"].fillna(
            obgyn_edges[f"{side}_other_{field}"]
        )

# --- Entity type for each side: OB-GYNs are always Individual (Entity Type 1) ---
for side in ["npi1", "npi2"]:
    obgyn_edges[f"{side}_entity"] = np.where(
        obgyn_edges[f"{side}_subspecialty"].notna(),
        "Individual",  # OB-GYN side is always individual
        obgyn_edges[f"{side}_entity_type"]  # Non-OB-GYN side from lookup
    )

# --- Slim down columns ---
keep_cols = [
    "npi1", "npi2", "paircount", "benecount", "samedaycount",
    "year", "days", "begdate", "enddate", "edge_type",
    "npi1_subspecialty", "npi1_other_type", "npi1_other_classification",
    "npi1_other_specialization", "npi1_specialty", "npi1_entity",
    "npi1_city_final", "npi1_state_final", "npi1_zip5_final",
    "npi2_subspecialty", "npi2_other_type", "npi2_other_classification",
    "npi2_other_specialization", "npi2_specialty", "npi2_entity",
    "npi2_city_final", "npi2_state_final", "npi2_zip5_final",
]
obgyn_edges = obgyn_edges[keep_cols].copy()

obgyn_edges = obgyn_edges.rename(columns={
    "npi1_city_final": "npi1_city", "npi1_state_final": "npi1_state", "npi1_zip5_final": "npi1_zip5",
    "npi2_city_final": "npi2_city", "npi2_state_final": "npi2_state", "npi2_zip5_final": "npi2_zip5",
})

# --- Report entity type breakdown ---
print(f"Full edge list: {len(obgyn_edges):,} rows x {len(obgyn_edges.columns)} cols")

# Entity type breakdown
both_individual = (obgyn_edges["npi1_entity"] == "Individual") & (obgyn_edges["npi2_entity"] == "Individual")
has_org = (obgyn_edges["npi1_entity"] == "Organization") | (obgyn_edges["npi2_entity"] == "Organization")

print(f"\nEntity type breakdown:")
print(f"  Both Individual:           {both_individual.sum():,} edges ({both_individual.sum()/len(obgyn_edges)*100:.1f}%)")
print(f"  At least one Organization: {has_org.sum():,} edges ({has_org.sum()/len(obgyn_edges)*100:.1f}%)")

# --- Filter to individual-to-individual only for network analysis ---
obgyn_edges_individuals = obgyn_edges[both_individual].copy()
obgyn_edges_orgs = obgyn_edges[~both_individual].copy()

print(f"\nIndividual-only edges (for network analysis): {len(obgyn_edges_individuals):,}")
print(f"Organization edges (excluded from network):     {len(obgyn_edges_orgs):,}")

print(f"\nTop 15 non-OB-GYN classifications (individual-to-individual only):")
other_side = pd.concat([
    obgyn_edges_individuals.loc[obgyn_edges_individuals["edge_type"]=="obgyn_other", "npi1_other_classification"],
    obgyn_edges_individuals.loc[obgyn_edges_individuals["edge_type"]=="obgyn_other", "npi2_other_classification"]
]).dropna()
print(other_side.value_counts().head(15).to_string())

display(obgyn_edges_individuals.head(5))

Full edge list: 243,052 rows x 28 cols

Entity type breakdown:
  Both Individual:           105,891 edges (43.6%)
  At least one Organization: 132,820 edges (54.6%)

Individual-only edges (for network analysis): 105,891
Organization edges (excluded from network):     137,161

Top 15 non-OB-GYN classifications (individual-to-individual only):
Radiology                 45854
Internal Medicine         18947
Pathology                  9059
Family Medicine            4852
Nurse Practitioner         3195
Specialist                 2894
Ophthalmology              1991
Emergency Medicine         1788
Physician Assistant        1563
Dermatology                1559
Podiatrist                 1348
Anesthesiology             1170
Surgery                     971
Urology                     835
Psychiatry & Neurology      691


,npi1,npi2,paircount,benecount,samedaycount,year,days,begdate,enddate,edge_type,...,npi1_zip5,npi2_subspecialty,npi2_other_type,npi2_other_classification,npi2_other_specialization,npi2_specialty,npi2_entity,npi2_city,npi2_state,npi2_zip5
1,1003000480,1548451727,30,14,1,2015,60,1/1/2015,10/1/2015,obgyn_other,...,80045,Obstetrics & Gynecology,NaN,NaN,NaN,Obstetrics & Gynecology,Individual,DENVER,CO,80222
2,1003001363,1205912847,111,17,1,2015,60,1/1/2015,10/1/2015,obgyn_other,...,92243,Female Pelvic Medicine and Reconstructive Surgery,NaN,NaN,NaN,Female Pelvic Medicine and Reconstructive Surgery,Individual,EL CENTRO,CA,92243
3,1003002627,1043246499,137,14,1,2015,60,1/1/2015,10/1/2015,obgyn_other,...,77566,NaN,Allopathic & Osteopathic Physicians,Anesthesiology,,Anesthesiology,Individual,ANGLETON,TX,77515
4,1003002627,1073518031,63,17,21,2015,60,1/1/2015,10/1/2015,obgyn_other,...,77566,NaN,Allopathic & Osteopathic Physicians,Pathology,Anatomic Pathology & Clinical Pathology,Pathology,Individual,CONROE,TX,77304
5,1003002627,1215977145,93,21,5,2015,60,1/1/2015,10/1/2015,obgyn_other,...,77566,NaN,Allopathic & Osteopathic Physicians,Internal Medicine,,Internal Medicine,Individual,LAKE JACKSON,TX,77566


In [19]:
# -------------------------------------------------------------------
# VALIDATE: Cross-state referrals check (now with full coverage)
# -------------------------------------------------------------------

both_states = obgyn_edges.dropna(subset=["npi1_state", "npi2_state"])
same_state = (both_states["npi1_state"] == both_states["npi2_state"]).sum()
cross_state = len(both_states) - same_state

print(f"Referral geography (where both providers have state info):")
print(f"  Same state:  {same_state:,} ({same_state/len(both_states)*100:.1f}%)")
print(f"  Cross state: {cross_state:,} ({cross_state/len(both_states)*100:.1f}%)")
print(f"  Total:       {len(both_states):,}")

Referral geography (where both providers have state info):
  Same state:  190,556 (79.8%)
  Cross state: 48,156 (20.2%)
  Total:       238,712


---

## 7. Compute County-Level Maternal Morbidity Rates

Join the two CDC WONDER files on `County of Residence Code` (FIPS) to compute:

$$\text{Morbidity Rate} = \frac{\text{Morbidity Births}}{\text{Total Births}}$$

Counties present in the births file but absent from the morbidity file are treated as having zero morbidity events (the count was either zero or suppressed below the reporting threshold).

In [20]:
# -------------------------------------------------------------------
# CLEAN: Prepare CDC WONDER data for join
# -------------------------------------------------------------------

def clean_cdc_data(df, value_col_name):
    """Clean a CDC WONDER DataFrame: drop notes, handle suppressed values."""
    out = df.copy()
    
    # Drop rows where County of Residence Code is missing or contains
    # 'Unidentified' grouping rows
    out = out[out["County of Residence Code"].notna()].copy()
    
    # Drop the Notes column (empty for data rows)
    if "Notes" in out.columns:
        out = out.drop(columns=["Notes"])
    
    # Convert Births to numeric; 'Suppressed' or non-numeric -> NaN
    out[value_col_name] = pd.to_numeric(out["Births"], errors="coerce")
    out = out.drop(columns=["Births"])
    
    # Rename for clarity
    out = out.rename(columns={
        "County of Residence": "county_name",
        "County of Residence Code": "fips"
    })
    
    return out

births_clean = clean_cdc_data(cdc_births, "total_births")
morbidity_clean = clean_cdc_data(cdc_morbidity, "morbidity_births")

print(f"Births clean:    {len(births_clean):,} rows")
print(f"Morbidity clean: {len(morbidity_clean):,} rows")

# Show suppressed counts
births_suppressed = births_clean["total_births"].isna().sum()
morbidity_suppressed = morbidity_clean["morbidity_births"].isna().sum()
print(f"\nSuppressed values:")
print(f"  Births:    {births_suppressed} counties")
print(f"  Morbidity: {morbidity_suppressed} counties")

Births clean:    626 rows
Morbidity clean: 577 rows

Suppressed values:
  Births:    0 counties
  Morbidity: 0 counties


In [21]:
# -------------------------------------------------------------------
# JOIN: Compute morbidity rate by county
# -------------------------------------------------------------------

county_rates = births_clean.merge(
    morbidity_clean[["fips", "morbidity_births"]],
    on="fips",
    how="left"
)

# Counties in births but not morbidity -> 0 morbidity births
county_rates["morbidity_births"] = county_rates["morbidity_births"].fillna(0)

# Compute rate (only where total_births is valid)
county_rates["morbidity_rate"] = (
    county_rates["morbidity_births"] / county_rates["total_births"]
)

# Extract state abbreviation from county name (e.g., 'Baldwin County, AL' -> 'AL')
county_rates["state"] = county_rates["county_name"].str.extract(r",\s*([A-Z]{2})$")

print(f"County morbidity rates: {len(county_rates):,} counties")
print(f"\nMorbidity rate statistics:")
print(county_rates["morbidity_rate"].describe().to_string())

print(f"\nTop 10 counties by morbidity rate:")
top10 = county_rates.nlargest(10, "morbidity_rate")[
    ["county_name", "fips", "total_births", "morbidity_births", "morbidity_rate"]
]
display(top10)

print(f"\nBottom 10 counties by morbidity rate (excluding zero):")
bottom10 = county_rates[county_rates["morbidity_rate"] > 0].nsmallest(10, "morbidity_rate")[
    ["county_name", "fips", "total_births", "morbidity_births", "morbidity_rate"]
]
display(bottom10)

County morbidity rates: 626 counties

Morbidity rate statistics:
count    626.000000
mean       0.013050
std        0.008382
min        0.000000
25%        0.007919
50%        0.012194
75%        0.016830
max        0.076811

Top 10 counties by morbidity rate:


,county_name,fips,total_births,morbidity_births,morbidity_rate
487,"Schuylkill County, PA",42107,1367,105.0,0.076811
351,"Dona Ana County, NM",35013,2839,160.0,0.056358
593,"Benton County, WA",53005,2759,145.0,0.052555
380,"Suffolk County, NY",36103,15559,809.0,0.051996
361,"Erie County, NY",36029,9916,448.0,0.045180
242,"Carroll County, MD",24013,1737,75.0,0.043178
367,"Niagara County, NY",36063,2139,86.0,0.040206
462,"Beaver County, PA",42007,1676,67.0,0.039976
413,"Cass County, ND",38017,2639,105.0,0.039788
359,"Chautauqua County, NY",36013,1345,52.0,0.038662



Bottom 10 counties by morbidity rate (excluding zero):


,county_name,fips,total_births,morbidity_births,morbidity_rate
579,"Fairfax County, VA",51059,14525,32.0,0.002203
114,"Osceola County, FL",12097,4328,10.0,0.002311
94,"Brevard County, FL",12009,5273,16.0,0.003034
63,"Stanislaus County, CA",06099,7862,24.0,0.003053
555,"Nueces County, TX",48355,4738,15.0,0.003166
109,"Marion County, FL",12083,3472,11.0,0.003168
36,"Fresno County, CA",06019,15129,50.0,0.003305
92,"Alachua County, FL",12001,2862,10.0,0.003494
123,"Seminole County, FL",12117,4753,17.0,0.003577
540,"Fort Bend County, TX",48157,9976,37.0,0.003709


---

## 8. Export Clean Datasets

Save all analysis-ready datasets to CSV for downstream analysis and visualization.

In [22]:
# -------------------------------------------------------------------
# EXPORT: Save analysis-ready files
# -------------------------------------------------------------------

# 1. OB-GYN Providers
obgyn_providers.to_csv(OBGYN_PROVIDERS_OUT, index=False)
print(f"Saved: {OBGYN_PROVIDERS_OUT}")
print(f"  {len(obgyn_providers):,} rows x {len(obgyn_providers.columns)} cols")
print(f"  Size: {OBGYN_PROVIDERS_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 2. Non-OB-GYN Providers (with entity_type column)
other_providers.to_csv(OTHER_PROVIDERS_OUT, index=False)
print(f"Saved: {OTHER_PROVIDERS_OUT}")
print(f"  {len(other_providers):,} rows x {len(other_providers.columns)} cols")
print(f"  Size: {OTHER_PROVIDERS_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 3. Full OB-GYN Edges (all entity types — for reference)
obgyn_edges.to_csv(OBGYN_EDGES_OUT, index=False)
print(f"Saved: {OBGYN_EDGES_OUT}")
print(f"  {len(obgyn_edges):,} rows x {len(obgyn_edges.columns)} cols")
print(f"  Size: {OBGYN_EDGES_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 4. Individual-only edges (for network analysis in Phase 4+)
OBGYN_EDGES_IND_OUT = OUTPUT_DIR / "obgyn_edges_individuals.csv"
obgyn_edges_individuals.to_csv(OBGYN_EDGES_IND_OUT, index=False)
print(f"Saved: {OBGYN_EDGES_IND_OUT}")
print(f"  {len(obgyn_edges_individuals):,} rows x {len(obgyn_edges_individuals.columns)} cols")
print(f"  Size: {OBGYN_EDGES_IND_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 5. County Morbidity Rates
county_rates.to_csv(COUNTY_MORBIDITY_OUT, index=False)
print(f"Saved: {COUNTY_MORBIDITY_OUT}")
print(f"  {len(county_rates):,} rows x {len(county_rates.columns)} cols")
print(f"  Size: {COUNTY_MORBIDITY_OUT.stat().st_size / 1024:.1f} KB")

Saved: ../data/outputs/obgyn_providers.csv
  60,422 rows x 9 cols
  Size: 5.1 MB

Saved: ../data/outputs/other_providers.csv
  46,527 rows x 13 cols
  Size: 6.0 MB

Saved: ../data/outputs/obgyn_edges.csv
  243,052 rows x 28 cols
  Size: 58.4 MB

Saved: ../data/outputs/obgyn_edges_individuals.csv
  105,891 rows x 28 cols
  Size: 25.8 MB

Saved: ../data/outputs/county_morbidity_rates.csv
  626 rows x 6 cols
  Size: 37.1 KB


In [23]:
# -------------------------------------------------------------------
# EXPORT: Save analysis-ready files
# -------------------------------------------------------------------

# 1. OB-GYN Providers
obgyn_providers.to_csv(OBGYN_PROVIDERS_OUT, index=False)
print(f"Saved: {OBGYN_PROVIDERS_OUT}")
print(f"  {len(obgyn_providers):,} rows x {len(obgyn_providers.columns)} cols")
print(f"  Size: {OBGYN_PROVIDERS_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 2. OB-GYN Edges (enriched)
obgyn_edges.to_csv(OBGYN_EDGES_OUT, index=False)
print(f"Saved: {OBGYN_EDGES_OUT}")
print(f"  {len(obgyn_edges):,} rows x {len(obgyn_edges.columns)} cols")
print(f"  Size: {OBGYN_EDGES_OUT.stat().st_size / (1024**2):.1f} MB")
print()

# 3. County Morbidity Rates
county_rates.to_csv(COUNTY_MORBIDITY_OUT, index=False)
print(f"Saved: {COUNTY_MORBIDITY_OUT}")
print(f"  {len(county_rates):,} rows x {len(county_rates.columns)} cols")
print(f"  Size: {COUNTY_MORBIDITY_OUT.stat().st_size / 1024:.1f} KB")

Saved: ../data/outputs/obgyn_providers.csv
  60,422 rows x 9 cols
  Size: 5.1 MB

Saved: ../data/outputs/obgyn_edges.csv
  243,052 rows x 28 cols
  Size: 58.4 MB

Saved: ../data/outputs/county_morbidity_rates.csv
  626 rows x 6 cols
  Size: 37.1 KB


---

## Phase 2 Summary

In [24]:
# -------------------------------------------------------------------
# SUMMARY
# -------------------------------------------------------------------

print("=" * 75)
print("PHASE 2 — DATA PREPARATION SUMMARY")
print("=" * 75)

print(f"\nInputs processed:")
print(f"  NPPES:       ~{NPPES_FILE.stat().st_size / (1024**3):.0f} GB scanned (two passes)")
print(f"  CMS:         ~{CMS_FILE.stat().st_size / (1024**3):.0f} GB scanned")
print(f"  NUCC:        {len(nucc_df):,} taxonomy codes")
print(f"  CDC WONDER:  {len(cdc_births):,} + {len(cdc_morbidity):,} county records")

print(f"\nOutputs:")
print(f"  {'File':<40} {'Rows':>10} {'Cols':>6}")
print(f"  {'-'*60}")

outputs_summary = [
    ("obgyn_providers.csv", obgyn_providers),
    ("other_providers.csv", other_providers),
    ("obgyn_edges.csv (all)", obgyn_edges),
    ("obgyn_edges_individuals.csv", obgyn_edges_individuals),
    ("county_morbidity_rates.csv", county_rates),
]

for name, df in outputs_summary:
    print(f"  {name:<40} {len(df):>10,} {len(df.columns):>6}")

print(f"\nKey statistics:")
print(f"  OB-GYN providers:              {len(obgyn_providers):,}")
print(f"  Non-OB-GYN providers:          {len(other_providers):,}")
print(f"    Individuals:                 {(other_providers['entity_type']=='Individual').sum():,}")
print(f"    Organizations:               {(other_providers['entity_type']=='Organization').sum():,}")
print(f"  Total edges (all):             {len(obgyn_edges):,}")
print(f"  Individual-only edges:         {len(obgyn_edges_individuals):,}")
print(f"    OB-GYN ↔ OB-GYN:            {(obgyn_edges_individuals['edge_type']=='obgyn_obgyn').sum():,}")
print(f"    OB-GYN ↔ Other Individual:   {(obgyn_edges_individuals['edge_type']=='obgyn_other').sum():,}")
print(f"  Organization edges (excluded): {len(obgyn_edges_orgs):,}")
print(f"  Counties with rates:           {county_rates['morbidity_rate'].notna().sum():,}")

print(f"\n{'='*75}")
print(f"All outputs saved to: {OUTPUT_DIR.resolve()}")
print(f"Network analysis (Phase 4+) will use obgyn_edges_individuals.csv")
print(f"Ready to proceed to Phase 3: EDA & Visualization (03_eda_visualization.ipynb)")

PHASE 2 — DATA PREPARATION SUMMARY

Inputs processed:
  NPPES:       ~10 GB scanned (two passes)
  CMS:         ~3 GB scanned
  NUCC:        838 taxonomy codes
  CDC WONDER:  627 + 578 county records

Outputs:
  File                                           Rows   Cols
  ------------------------------------------------------------
  obgyn_providers.csv                          60,422      9
  other_providers.csv                          46,527     13
  obgyn_edges.csv (all)                       243,052     28
  obgyn_edges_individuals.csv                 105,891     28
  county_morbidity_rates.csv                      626      6

Key statistics:
  OB-GYN providers:              60,422
  Non-OB-GYN providers:          46,527
    Individuals:                 37,114
    Organizations:               8,031
  Total edges (all):             243,052
  Individual-only edges:         105,891
    OB-GYN ↔ OB-GYN:            3,751
    OB-GYN ↔ Other Individual:   102,140
  Organization edges (ex

In [25]:
# -------------------------------------------------------------------
# EXPORT: Summary statistics table for Tableau
# -------------------------------------------------------------------

n_other_individual = (other_providers["entity_type"] == "Individual").sum()
n_other_org = (other_providers["entity_type"] == "Organization").sum()

summary_rows = [
    {"Category": "Data Scope", "Metric": "OB-GYN Providers", "Value": len(obgyn_providers), "Detail": "Individual providers with 207V taxonomy code"},
    {"Category": "Data Scope", "Metric": "Non-OB-GYN Individuals", "Value": n_other_individual, "Detail": "Individual providers sharing patients with OB-GYNs"},
    {"Category": "Data Scope", "Metric": "Non-OB-GYN Organizations", "Value": n_other_org, "Detail": "Facilities, hospitals, labs sharing patients with OB-GYNs"},
    {"Category": "Data Scope", "Metric": "Total Providers in Network", "Value": len(obgyn_providers) + len(other_providers), "Detail": "All unique providers (individuals + organizations) in the edge list"},
    {"Category": "Data Scope", "Metric": "OB-GYN Subspecialties", "Value": obgyn_providers["subspecialty"].nunique(), "Detail": "Distinct 207V taxonomy classifications"},
    {"Category": "Data Scope", "Metric": "Non-OB-GYN Classifications", "Value": other_providers["classification"].nunique(), "Detail": "Distinct NUCC classifications for other providers"},
    {"Category": "Network", "Metric": "Total Edges (all)", "Value": len(obgyn_edges), "Detail": "All provider pairs sharing >= 11 Medicare patients within 60 days"},
    {"Category": "Network", "Metric": "Individual-to-Individual Edges", "Value": len(obgyn_edges_individuals), "Detail": "Edges used for network analysis (organizations excluded)"},
    {"Category": "Network", "Metric": "OB-GYN to OB-GYN Edges", "Value": (obgyn_edges_individuals["edge_type"] == "obgyn_obgyn").sum(), "Detail": "Both providers are OB-GYN individuals"},
    {"Category": "Network", "Metric": "OB-GYN to Other Individual Edges", "Value": (obgyn_edges_individuals["edge_type"] == "obgyn_other").sum(), "Detail": "OB-GYN paired with a different individual specialist"},
    {"Category": "Outcomes", "Metric": "Counties with Morbidity Data", "Value": county_rates["morbidity_rate"].notna().sum(), "Detail": "CDC WONDER 2016 county-level maternal morbidity"},
    {"Category": "Outcomes", "Metric": "Total Births Tracked", "Value": int(county_rates["total_births"].sum()), "Detail": "Across all counties in the dataset"},
]

summary_df = pd.DataFrame(summary_rows)
summary_df["Value_Formatted"] = summary_df["Value"].apply(lambda x: f"{x:,}")

SUMMARY_OUT = OUTPUT_DIR / "phase2_summary_stats.csv"
summary_df.to_csv(SUMMARY_OUT, index=False)
print(f"Saved: {SUMMARY_OUT}")
display(summary_df[["Category", "Metric", "Value_Formatted", "Detail"]])

Saved: ../data/outputs/phase2_summary_stats.csv


,Category,Metric,Value_Formatted,Detail
0,Data Scope,OB-GYN Providers,"60,422",Individual providers with 207V taxonomy code
1,Data Scope,Non-OB-GYN Individuals,"37,114",Individual providers sharing patients with OB-...
2,Data Scope,Non-OB-GYN Organizations,"8,031","Facilities, hospitals, labs sharing patients w..."
3,Data Scope,Total Providers in Network,"106,949",All unique providers (individuals + organizati...
4,Data Scope,OB-GYN Subspecialties,10,Distinct 207V taxonomy classifications
5,Data Scope,Non-OB-GYN Classifications,134,Distinct NUCC classifications for other providers
6,Network,Total Edges (all),"243,052",All provider pairs sharing >= 11 Medicare pati...
7,Network,Individual-to-Individual Edges,"105,891",Edges used for network analysis (organizations...
8,Network,OB-GYN to OB-GYN Edges,"3,751",Both providers are OB-GYN individuals
9,Network,OB-GYN to Other Individual Edges,"102,140",OB-GYN paired with a different individual spec...


---

## Next Steps

Proceed to **`03_eda_visualization.ipynb`**:
1. Distribution analysis — provider counts by state, sub-specialty, edge weight distributions
2. Geographic coverage maps — where are OB-GYN providers concentrated vs. sparse?
3. Network summary statistics — degree distributions, connected components
4. Morbidity rate exploration — geographic variation, relationship to provider density
5. Correlation analysis — early signal detection between network features and health outcomes